# GIRF - Dyn-method with triangular pulses

Estimate the gradient impulse response function using the Dyn-method with triangular pulses

### Imports

In [ ]:
import tempfile
from pathlib import Path

import matplotlib.pyplot as plt
import MRzeroCore as mr0
import numpy as np
import torch
from mrpro.data import KData
from mrpro.data.traj_calculators import KTrajectoryCartesian

from mrseq.sequences.girf_dyn_triangle import main as create_seq
from mrseq.utils import sys_defaults
from mrseq.utils.Gmtf import Gmtf
from mrseq.utils.Gmtf import build_input_triangles
from mrseq.utils.Gmtf import estimate_gmtf
from mrseq.utils.Gmtf import phase_to_gradient
from mrseq.utils.Gmtf import unwrap_phase_difference

### Settings
We are going to use a numerical phantom with a matrix size of 128 x 128. 

In [ ]:
image_matrix_size = [128, 128]

tmp = tempfile.TemporaryDirectory()
fname_mrd = Path(tmp.name) / 'girf_dyn_triangle.mrd'

### Create the digital phantom

We use the standard Brainweb phantom from [MRzero](https://github.com/MRsources/MRzero-Core).

In [ ]:
phantom = mr0.util.load_phantom(image_matrix_size)
phantom.T1[:] = 400e-3
phantom.T2[:] = 40e-3
phantom.B1[:] = 1.0
phantom.B0[:] = 0.0

### Create the GIRF Dyn sequence

To create the GIRF Dyn sequence, we use the previously imported [girf_dyn_triangle script](../src/mrseq/scripts/girf_dyn_triangle.py).

In [ ]:
sequence, fname_seq = create_seq(
    system=sys_defaults,
    test_report=False,
    timing_check=False,
)

### Simulate the sequence

Now, we pass the sequence and the phantom to the MRzero simulation and save the simulated signal as an (ISMR)MRD file.

In [ ]:
mr0_sequence = mr0.Sequence.import_file(str(fname_seq.with_suffix('.seq')))
signal, ktraj_adc = mr0.util.simulate(mr0_sequence, phantom, accuracy=1e-1)
mr0.sig_to_mrd(fname_mrd, signal, sequence)

### Estimate GIRF

Here we are doing some of steps manually to show the intermediate results. At the end we are going to calculate the GIRF 
or to be more accurate the GMTF (Gradient Modulation Transfer Function) directly from the raw k-space file and the sequency file.

#### Sort data and compress coils

In [ ]:
fname_mrd = Path(
    '/Users/kolbit01/Documents/Data/GIRF/mrseq_scaling/2026_09_16/meas_MID00124_FID08240_girf_original.mrd'
)
kdata = KData.from_file(fname_mrd, trajectory=KTrajectoryCartesian())
idx = np.lexsort(
    (
        kdata.header.acq_info.idx.phase.squeeze(),
        kdata.header.acq_info.idx.repetition.squeeze(),
        kdata.header.acq_info.idx.average.squeeze(),
    )
)
kdata_sorted = kdata[torch.as_tensor(idx)]
kdata_sorted = kdata_sorted.rearrange(
    '(avg rep ph) ... -> avg rep ph ...',
    rep=int(kdata.header.acq_info.idx.repetition.max()) + 1,
    avg=int(kdata.header.acq_info.idx.average.max()) + 1,
    ph=int(kdata.header.acq_info.idx.phase.max()) + 1,
)
kdata_single_coil = kdata_sorted.compress_coils(n_compressed_coils=1).data.squeeze()

#### Get information about the scan from the seq file

In [ ]:
import pypulseq as pp

sequence = pp.Sequence()
fname_seq = '/Users/kolbit01/Documents/Data/GIRF/mrseq_scaling/2026_09_16/girf_dyn_triangle.seq'
sequence.read(fname_seq)

adc_duration = sequence.get_definition('AdcDuration')
dwell_time = sequence.get_definition('DwellTime')
slice_pos = sequence.get_definition('SlicePos')
gamma = sequence.system.gamma * 2 * torch.pi
rise_times = sequence.get_definition('RiseTimes')
slew_rate = sequence.get_definition('SlewRate') / sequence.system.gamma
g_delay = sequence.get_definition('GradientPreEmphasisDelay')
g_amplitude_coeff = sequence.get_definition('GradAmplitudeCoeff')

#### Unwrapped measured gradient shapes

In [ ]:
time = torch.linspace(0.0, adc_duration, kdata_single_coil.shape[-1])

phase = unwrap_phase_difference(kdata_single_coil)
phase_mean = phase.mean(dim=0)
phase_std = phase.std(dim=0)

grad_output_mean, grad_output_std = phase_to_gradient(phase_mean, phase_std, slice_pos, gamma, dwell_time)

In [ ]:
# Plot unwrapped phase (k-space trajectories) for each gradient axis
grad_axes = ['x', 'y', 'z']
fig, axes = plt.subplots(len(grad_axes), 1, sharex=True)  # (avg, ax, rise, samp)
for ax_idx, (ax, label) in enumerate(zip(axes, grad_axes, strict=True)):
    for rise_idx in range(phase.shape[2]):
        ax.plot(time, phase_mean.numpy()[ax_idx, rise_idx, :])
    ax.set_title(f'K-space — {label}')
    ax.set_xlabel('Time [s]')
    ax.grid()
fig.tight_layout()
plt.show()

#### Create ideal gradient triangles

In [ ]:
g_amplitude_coeff[0] *= -1

In [ ]:
grad_input = build_input_triangles(
    rise_times, slew_rate, g_delay, g_amplitude_coeff, dwell_time, grad_output_mean.shape[-1]
)

#### Compare ideal and measured gradient triangles

In [ ]:
# Overlay ideal input triangles against measured X-axis output
plt.figure()
for i in range(grad_input.shape[0]):
    line = plt.plot(time[:-1] * 1e3, grad_input[i, :], '-')
    plt.plot(time[:-1] * 1e3, grad_output_mean[0, i, :], color=line[0].get_color(), linestyle='dashed')

plt.xlim((0.0, 0.4))
plt.xlabel('Time [ms]')
plt.ylabel('Gradient Strength [T/m]')
plt.title('Input (solid) vs Measurement (dashed)')
plt.grid()
plt.tight_layout()
plt.show()

#### Calculate GIRF and plot

In [ ]:
gmtf = estimate_gmtf(grad_input, grad_output_mean)

In [ ]:
# Plot magnitude and phase of the GMTF for all axes
grad_axes = ['x', 'y', 'z']

frequency_vector = (1.0 / dwell_time) * torch.arange(-gmtf.shape[-1] // 2, gmtf.shape[-1] // 2) / (gmtf.shape[-1])
half = len(frequency_vector) // 2

fig, (ax1, ax2) = plt.subplots(1, 2, sharex=True, figsize=(10, 5))
for k, label in enumerate(grad_axes):
    ax1.plot(frequency_vector[half:] * 1e-3, abs(gmtf.numpy()[k, half:]), label=label)
    ax2.plot(frequency_vector[half:] * 1e-3, np.unwrap(np.angle(gmtf.numpy()[k, half:])), label=label)

ax1.set(xlabel='Frequency (kHz)', title='GMTF Magnitude', xlim=(0, 10), ylim=(0.8, 1.025))
ax2.set(xlabel='Frequency (kHz)', title='GMTF Phase', xlim=(0, 10), ylim=(-0.1, np.pi / 4))

for ax in (ax1, ax2):
    ax.legend()
    ax.grid(visible=True, which='major', color='#666666')
    ax.minorticks_on()
    ax.grid(visible=True, which='minor', color='#999999', linestyle='-', alpha=0.2)

fig.tight_layout()
plt.show()

#### Now we do it all in one step

In [ ]:
gmtf = Gmtf.compute_gmtf(fname_mrd, fname_seq)
gmtf.plot()